# BERT CV — 부담 예측 모델 평가

klue/bert-base fine-tuning, 5-fold GroupKFold (CO/PO).

- **런타임:** A100 GPU
- **데이터:** `data.zip` 업로드
- **선행:** `tfidf_cv.py` 결과가 `results/`에 있어야 비교표 생성 가능


In [1]:
# Dependencies + GPU check
!pip install -q transformers torch scikit-learn scipy pandas tabulate

import torch
print(f"torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {DEVICE}")

torch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Using: cuda


In [2]:
# Data upload and preprocessing
import os, zipfile
from google.colab import files

if not os.path.exists("/content/raw_everytime_reviews.csv"):
    uploaded = files.upload()  # bert_cv_data.zip 업로드
    with zipfile.ZipFile("/content/bert_cv_data.zip") as z:
        z.extractall("/content/")

import pandas as pd
import numpy as np

RAW_PATH = "/content/raw_everytime_reviews.csv"
COURSES_PATH = "/content/courses.csv"
TARGETS = ["workload_label", "teamwork_load_label", "grading_strictness_label"]

def clean_text(x):
    return "" if pd.isna(x) else str(x).strip()

def make_course_key(df):
    return df["course_name"].astype(str).str.strip() + "__" + df["professor"].fillna("").astype(str).str.strip()

raw = pd.read_csv(RAW_PATH)
courses = pd.read_csv(COURSES_PATH)
raw["course_key"] = make_course_key(raw)
courses["course_key"] = make_course_key(courses)

ct = courses[["course_key", *TARGETS]].copy()
for col in TARGETS:
    ct[col] = pd.to_numeric(ct[col], errors="coerce")
ct = ct.groupby("course_key", as_index=False)[TARGETS].mean().dropna()
ct = ct.rename(columns={c: f"target_{c}" for c in TARGETS})
TARGET_COLS = [f"target_{c}" for c in TARGETS]

data = raw.merge(ct, on="course_key", how="inner")
data["raw_review_text"] = data["raw_review_text"].apply(clean_text)
data = data[data["raw_review_text"].str.len() > 0].reset_index(drop=True)

rpc = data.groupby("course_key").size()
print(f"리뷰 {len(data):,}건 | 과목 {data['course_key'].nunique()} | 교수 {data['professor'].fillna('').nunique()}")
print(f"과목당 리뷰: min={rpc.min()} med={rpc.median():.0f} max={rpc.max()} mean={rpc.mean():.1f}")

Saving bert_cv_data.zip to bert_cv_data.zip
리뷰 18,787건 | 과목 315 | 교수 245
과목당 리뷰: min=1 med=65 max=100 mean=59.6


In [3]:
# Model definitions
import math, time, json, gc
from pathlib import Path
from dataclasses import dataclass, asdict

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold

class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else np.asarray(labels, dtype=np.float32)
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        if self.labels is None: return self.texts[idx]
        return self.texts[idx], self.labels[idx]

class BertRegressor(nn.Module):
    def __init__(self, model_name, output_dim, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, output_dim)
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None: kw["token_type_ids"] = token_type_ids
        out = self.bert(**kw)
        pooled = out.pooler_output if hasattr(out, "pooler_output") and out.pooler_output is not None else out.last_hidden_state[:, 0, :]
        return 1.0 + 4.0 * torch.sigmoid(self.head(self.dropout(pooled)))

def make_collate(tokenizer, max_len, has_labels):
    def fn(batch):
        if has_labels:
            texts, labels = zip(*batch)
            enc = tokenizer(list(texts), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
            return enc, torch.tensor(np.asarray(labels, dtype=np.float32))
        enc = tokenizer(list(batch), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        return enc
    return fn

def safe_pearson(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) < 2 or np.std(a) < 1e-8 or np.std(b) < 1e-8: return np.nan
    return float(pearsonr(a, b).statistic)

print("Model definitions loaded.")

Model definitions loaded.


In [6]:
# Training and CV functions

def train_bert_fold(train_df, val_df, hparams, device, fold_name="Fold"):
    """한 fold 학습. 과목 단위 (y_true, y_pred) 반환."""
    t0 = time.time()
    model_name = hparams["model_name"]
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_labels = train_df[TARGET_COLS].values.astype(np.float32)
    val_labels = val_df[TARGET_COLS].values.astype(np.float32)

    train_loader = DataLoader(
        ReviewDataset(train_df["raw_review_text"].values, train_labels),
        batch_size=hparams["batch_size"], shuffle=True,
        collate_fn=make_collate(tokenizer, hparams["max_length"], True),
    )
    val_loader = DataLoader(
        ReviewDataset(val_df["raw_review_text"].values, val_labels),
        batch_size=hparams["batch_size"], shuffle=False,
        collate_fn=make_collate(tokenizer, hparams["max_length"], True),
    )

    model = BertRegressor(model_name, len(TARGETS), hparams["dropout"]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=hparams["lr"], weight_decay=0.01)
    total_steps = len(train_loader) * hparams["epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)
    criterion = nn.MSELoss()

    best_val, best_state = float("inf"), None
    for epoch in range(1, hparams["epochs"] + 1):
        model.train()
        train_loss, n = 0.0, 0
        for enc, labels in train_loader:
            enc = {k: v.to(device) for k, v in enc.items()}
            labels = labels.to(device)
            optimizer.zero_grad()
            pred = model(**enc)
            loss = criterion(pred, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_loss += float(loss.detach()) * labels.shape[0]
            n += labels.shape[0]

        model.eval()
        val_loss, vn = 0.0, 0
        with torch.no_grad():
            for enc, labels in val_loader:
                enc = {k: v.to(device) for k, v in enc.items()}
                labels = labels.to(device)
                val_loss += float(criterion(model(**enc), labels)) * labels.shape[0]
                vn += labels.shape[0]

        t_mse, v_mse = train_loss/n, val_loss/vn
        print(f"  {fold_name} ep{epoch}/{hparams['epochs']} train={t_mse:.4f} val={v_mse:.4f} ({time.time()-t0:.0f}s)")
        if v_mse < best_val:
            best_val = v_mse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state: model.load_state_dict(best_state)

    # 과목 단위 예측
    model.eval()
    preds = []
    with torch.no_grad():
        for enc, _ in val_loader:
            enc = {k: v.to(device) for k, v in enc.items()}
            preds.append(model(**enc).cpu().numpy())
    review_preds = np.clip(np.vstack(preds), 1.0, 5.0)

    # 과목 단위 집계
    pred_df = pd.DataFrame({"course_key": val_df["course_key"].values})
    for i, t in enumerate(TARGETS):
        pred_df[f"pred_{t}"] = review_preds[:, i]
        pred_df[f"true_{t}"] = val_df[TARGET_COLS[i]].values
    course_df = pred_df.groupby("course_key").agg(
        {**{f"pred_{t}": "mean" for t in TARGETS}, **{f"true_{t}": "first" for t in TARGETS}}
    ).reset_index()

    y_true = course_df[[f"true_{t}" for t in TARGETS]].values
    y_pred = course_df[[f"pred_{t}" for t in TARGETS]].values

    del model, tokenizer, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return y_true, y_pred, time.time() - t0


def run_bert_cv(data, hparams, n_splits=5, split_axis="course"):
    """두 축 GroupKFold CV."""
    course_frame = data[["course_key", "professor"]].drop_duplicates("course_key").reset_index(drop=True)
    keys = course_frame["course_key"].values
    groups = keys if split_axis == "course" else course_frame["professor"].fillna("").str.strip().values

    gkf = GroupKFold(n_splits=n_splits)
    all_metrics = []
    cv_t0 = time.time()

    for fold, (tr_i, va_i) in enumerate(gkf.split(keys, groups=groups), 1):
        tr_set, va_set = set(keys[tr_i]), set(keys[va_i])
        tr_df = data[data["course_key"].isin(tr_set)].reset_index(drop=True)
        va_df = data[data["course_key"].isin(va_set)].reset_index(drop=True)
        print(f"\nFold {fold}/{n_splits}: train {len(tr_df)} reviews ({tr_df['course_key'].nunique()} courses), val {len(va_df)} ({va_df['course_key'].nunique()})")

        y_true, y_pred, elapsed = train_bert_fold(tr_df, va_df, hparams, DEVICE, f"F{fold}")
        baseline = np.tile(y_true.mean(axis=0), (len(y_true), 1))

        for label, yp in [("baseline", baseline), ("bert", y_pred)]:
            for i, t in enumerate(TARGETS):
                all_metrics.append({"fold": fold, "target": t, "model": label,
                    "mae": mean_absolute_error(y_true[:,i], yp[:,i]),
                    "rmse": math.sqrt(mean_squared_error(y_true[:,i], yp[:,i])),
                    "pearson_r": safe_pearson(y_true[:,i], yp[:,i])})
            all_metrics.append({"fold": fold, "target": "average", "model": label,
                "mae": mean_absolute_error(y_true.ravel(), yp.ravel()),
                "rmse": math.sqrt(mean_squared_error(y_true.ravel(), yp.ravel())),
                "pearson_r": safe_pearson(y_true.ravel(), yp.ravel())})

        total = time.time() - cv_t0
        eta = total / fold * (n_splits - fold)
        print(f"  fold {fold} done {elapsed:.0f}s (total {total:.0f}s, ETA {eta:.0f}s)")

    return pd.DataFrame(all_metrics)

print("CV functions loaded.")

CV functions loaded.


In [7]:
# Baseline: leave-course-out (5-fold x 3 epochs)
FAST_HP_CO = {
    "model_name": "klue/bert-base",
    "epochs": 3,
    "batch_size": 128,
    "max_length": 128,
    "lr": 2e-5,
    "dropout": 0.1,
}

print("=" * 60)
print("BASELINE: leave-course-out (batch=128)")
print("=" * 60)
bl_co = run_bert_cv(data, FAST_HP_CO, n_splits=5, split_axis="course")

bl_co_summary = bl_co.groupby(["model", "target"], as_index=False).agg(
    MAE_mean=("mae", "mean"), MAE_std=("mae", lambda x: x.std(ddof=0)),
    RMSE_mean=("rmse", "mean"), Pearson_mean=("pearson_r", "mean"),
    Pearson_std=("pearson_r", lambda x: x.std(ddof=0)),
)
print("\n" + bl_co_summary.to_string(index=False))

BASELINE: leave-course-out

Fold 1/5: train 14422 reviews (252 courses), val 4365 (63)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1 ep1/3 train=0.6243 val=0.4907 (70s)
  F1 ep2/3 train=0.4320 val=0.4690 (138s)
  F1 ep3/3 train=0.3848 val=0.4768 (207s)
  fold 1 done 214s (total 214s, ETA 854s)

Fold 2/5: train 15084 reviews (252 courses), val 3703 (63)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F4 ep1/3 train=0.6269 val=0.5195 (71s)
  F4 ep2/3 train=0.4197 val=0.5131 (141s)
  F4 ep3/3 train=0.3747 val=0.5042 (210s)
  fold 4 done 216s (total 871s, ETA 218s)

Fold 5/5: train 15046 reviews (252 courses), val 3741 (63)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F5 ep1/3 train=0.6607 val=0.5082 (72s)
  F5 ep2/3 train=0.4216 val=0.5086 (142s)
  F5 ep3/3 train=0.3783 val=0.5056 (212s)
  fold 5 done 218s (total 1089s, ETA 0s)

   model                   target  MAE_mean  MAE_std  RMSE_mean  Pearson_mean  Pearson_std
baseline                  average  0.696138 0.031496   0.866359      0.349221     0.031117
baseline grading_strictness_label  0.481641 0.046377   0.560396           NaN          NaN
baseline      teamwork_load_label  1.019129 0.021966   1.155826           NaN          NaN
baseline           workload_label  0.587644 0.054443   0.771618           NaN          NaN
    bert                  average  0.496751 0.019104   0.625735      0.744406     0.024975
    bert grading_strictness_label  0.438248 0.038303   0.527367      0.449829     0.115268
    bert      teamwork_load_label  0.608138 0.037804   0.742136      0.788782     0.039088
    bert           workload_label  0.443866 0.040460   0.583616      0.691195     0.071409


In [8]:
# Baseline: leave-professor-out (5-fold x 3 epochs)
import logging; logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

FAST_HP = {
    "model_name": "klue/bert-base",
    "epochs": 3,
    "batch_size": 128,
    "max_length": 128,
    "lr": 2e-5,
    "dropout": 0.1,
}

print("=" * 60)
print("BASELINE: leave-professor-out (batch=128)")
print("=" * 60)
bl_po = run_bert_cv(data, FAST_HP, n_splits=5, split_axis="professor")

bl_po_summary = bl_po.groupby(["model", "target"], as_index=False).agg(
    MAE_mean=("mae", "mean"), MAE_std=("mae", lambda x: x.std(ddof=0)),
    RMSE_mean=("rmse", "mean"), Pearson_mean=("pearson_r", "mean"),
    Pearson_std=("pearson_r", lambda x: x.std(ddof=0)),
)
print("\n" + bl_po_summary.to_string(index=False))

BASELINE: leave-professor-out (batch=128)

Fold 1/5: train 15264 reviews (252 courses), val 3523 (63)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1 ep1/3 train=0.6547 val=0.4830 (72s)
  F1 ep2/3 train=0.4304 val=0.4736 (143s)
  F1 ep3/3 train=0.3832 val=0.4634 (214s)
  fold 1 done 219s (total 219s, ETA 878s)

Fold 2/5: train 14574 reviews (252 courses), val 4213 (63)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F4 ep1/3 train=0.6258 val=0.5228 (72s)
  F4 ep2/3 train=0.4183 val=0.5254 (143s)
  F4 ep3/3 train=0.3720 val=0.5147 (213s)
  fold 4 done 219s (total 873s, ETA 218s)

Fold 5/5: train 14742 reviews (252 courses), val 4045 (63)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F5 ep1/3 train=0.6939 val=0.4653 (71s)
  F5 ep2/3 train=0.4546 val=0.4353 (140s)
  F5 ep3/3 train=0.4012 val=0.4356 (209s)
  fold 5 done 216s (total 1089s, ETA 0s)

   model                   target  MAE_mean  MAE_std  RMSE_mean  Pearson_mean  Pearson_std
baseline                  average  0.697154 0.033053   0.869039      0.346419     0.044321
baseline grading_strictness_label  0.468037 0.031742   0.554441           NaN          NaN
baseline      teamwork_load_label  1.036162 0.061206   1.163142           NaN          NaN
baseline           workload_label  0.587262 0.038412   0.776196           NaN          NaN
    bert                  average  0.492856 0.025263   0.619306      0.750054     0.053016
    bert grading_strictness_label  0.450403 0.031619   0.541249      0.414358     0.122267
    bert      teamwork_load_label  0.590511 0.054269   0.720270      0.807823     0.061721
    bert           workload_label  0.437653 0.015915   0.579828      0.697512     0.027327


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [10]:
# Save results + comparison with TF-IDF
import logging; logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
from pathlib import Path

RESULTS = Path("/content/results")
RESULTS.mkdir(exist_ok=True)

bl_co.to_csv(RESULTS / "bert_co_fold_metrics.csv", index=False, encoding="utf-8-sig")
bl_co_summary.to_csv(RESULTS / "bert_co_summary.csv", index=False, encoding="utf-8-sig")
bl_po.to_csv(RESULTS / "bert_po_fold_metrics.csv", index=False, encoding="utf-8-sig")
bl_po_summary.to_csv(RESULTS / "bert_po_summary.csv", index=False, encoding="utf-8-sig")

# Read TF-IDF results (from tfidf_cv.py output)
tfidf_co = pd.read_csv(RESULTS / "leave_course_out" / "baseline_summary.csv")
tfidf_po = pd.read_csv(RESULTS / "leave_professor_out" / "baseline_summary.csv")

ms = lambda m,s: f"{m:.3f} ± {s:.3f}" if not pd.isna(s) else f"{m:.3f}"
tmap = {"workload_label":"workload","teamwork_load_label":"teamwork",
        "grading_strictness_label":"grading","average":"average"}

rows = []
for t in ["workload_label","teamwork_load_label","grading_strictness_label","average"]:
    s = tmap[t]
    tc = tfidf_co[(tfidf_co["model"]=="tfidf_mlp")&(tfidf_co["target"]==t)].iloc[0]
    bc = bl_co_summary[(bl_co_summary["model"]=="bert")&(bl_co_summary["target"]==t)].iloc[0]
    tp = tfidf_po[(tfidf_po["model"]=="tfidf_mlp")&(tfidf_po["target"]==t)].iloc[0]
    bp = bl_po_summary[(bl_po_summary["model"]=="bert")&(bl_po_summary["target"]==t)].iloc[0]
    rows.append({
        "Target": s,
        "TF-IDF CO MAE": ms(tc["MAE_mean"],tc["MAE_std"]),
        "BERT CO MAE": ms(bc["MAE_mean"],bc["MAE_std"]),
        "TF-IDF CO r": ms(tc["Pearson_mean"],tc["Pearson_std"]),
        "BERT CO r": ms(bc["Pearson_mean"],bc["Pearson_std"]),
        "TF-IDF PO MAE": ms(tp["MAE_mean"],tp["MAE_std"]),
        "BERT PO MAE": ms(bp["MAE_mean"],bp["MAE_std"]),
    })

compare = pd.DataFrame(rows)
compare.to_csv(RESULTS / "full_comparison.csv", index=False, encoding="utf-8-sig")
print("TF-IDF+MLP vs BERT\n")
print(compare.to_markdown(index=False))
